# Food Segmentation + Depth Fusion + Recommendation Pipeline (v3.3)

**Run all cells top-to-bottom in one Colab session.**

## Flow
Install -> Download dataset -> Train YOLOv8s-seg -> Load model ->
Segment -> Depth (MiDaS) -> Volume/Weight -> Nutrition DB ->
Diabetic Plate Assessment -> Recommendation

## What is new in v3.1 (from v3.0)
- **Dual-mode depth calibration**: relative (percentile) + calibrated (absolute).
  Toggle via `USE_CALIBRATED_DEPTH`.
- **`calibrate_depth()`** routine for one-time calibration using a known object.
- **Unified NUTRITION_DB**: `FOOD_DENSITIES` dict removed; density read from NUTRITION_DB.
- **Per-detection mask** carried through pipeline (fixes duplicate class collision).
- **Mask cleanup fallback** chain: full → eroded_only → raw (fixes plantain dropout).
- **Detection output enriched** with GI, carbs, mask for recommendation engine.
- **Starch subtype split**: `starch_moulded` vs `starch_spread`. Orange volume
  reference only applies to moulded staples (fufu, banku). Spread starches
  assessed by plate ratio only.
- **`soup_sauce` category**: okro soup, light soup, shito excluded from plate ratio.
- **STARCH_REFERENCE_VOLUME** updated to 220 cm³ (calibrated from dietician photo).
- **NCNN export** (not ONNX) as primary format for Raspberry Pi.
- **Plate diameter** set to 25 cm (consistent with thesis FR-05).
- **Field naming unified**: `class_name` used consistently throughout pipeline.

## v2.6 pipeline retained intact
- 3-tier plate detection (YOLO -> Hough -> depth histogram)
- MiDaS height estimation (percentile normalisation in relative mode)
- `clean_food_mask()`, `detect_plate()`, `estimate_weight()`, `visualize_debug()`
- Calibrated depth mode (optional)

## 1. Install Dependencies

In [ ]:
# torch and torchvision are intentionally NOT listed here.
# Colab pre-installs CUDA-enabled torch; pip-installing them would replace
# it with a CPU-only build.
!pip install ultralytics opencv-python pillow numpy matplotlib roboflow timm -q

## 2. Imports

In [ ]:
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import json
import os
from pathlib import Path
from typing import Optional, List, Dict, Any
from ultralytics import YOLO

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

## 3. Download Dataset (Roboflow v3)

In [ ]:
from roboflow import Roboflow

rf      = Roboflow(api_key='Sbqm20sMl8b8VAsblgEE')
project = rf.workspace('paayaw').project('food-segmentation-rdkme')
dataset = project.version(3).download('yolov8')

DATA_YAML = os.path.join(os.path.abspath(dataset.location), 'data.yaml')
VALID_DIR = Path(dataset.location) / 'valid' / 'images'

print(f'Dataset  : {dataset.location}')
print(f'data.yaml: {DATA_YAML}')
print(f'Exists   : {os.path.exists(DATA_YAML)}')
print(f'Contents : {os.listdir(dataset.location)}')

## 4. Train YOLOv8s-seg

Using **yolov8s-seg** (small) — better accuracy than nano at an acceptable
Colab training cost. If T4 runs out of memory reduce `BATCH_SIZE` to 8.

In [ ]:
EPOCHS     = 50
IMAGE_SIZE = 640
BATCH_SIZE = 16
MODEL_NAME = 'ghanaian_food_seg_v3'

seg_model = YOLO('yolov8s-seg.pt')   # s = small; better than nano
seg_model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    name=MODEL_NAME,
    patience=10,
    device=DEVICE,
    save=True,
    plots=True,
    exist_ok=True,
    verbose=True,
)

MODEL_PATH = f'runs/segment/{MODEL_NAME}/weights/best.pt'
print(f'Model saved to: {MODEL_PATH}')

## 5. Load Trained Model

In [ ]:
if not Path(MODEL_PATH).exists():
    raise FileNotFoundError(
        f'Model not found at {MODEL_PATH}. Run the training cell first.'
    )

seg_model   = YOLO(MODEL_PATH)
CLASS_NAMES = list(seg_model.names.values())
PLATE_CLASS = next((c for c in CLASS_NAMES if c.lower() == 'plate'), None)

print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')
print(f'Plate class : {PLATE_CLASS}')

## 6. Test Segmentation on Validation Images

In [ ]:
CONFIDENCE       = 0.25
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
test_images      = sorted(
    p for p in VALID_DIR.glob('*') if p.suffix.lower() in IMAGE_EXTENSIONS
)[:6]
print(f'Testing on {len(test_images)} images from {VALID_DIR}')

all_results = []
for img_path in test_images:
    image = cv2.imread(str(img_path))
    if image is None:
        print(f'  Skipping unreadable: {img_path.name}')
        continue
    result     = seg_model.predict(image, conf=CONFIDENCE, verbose=False)[0]
    detections = []
    if result.masks is not None:
        for box, mask in zip(result.boxes, result.masks.data.cpu().numpy()):
            cls_id = int(box.cls)
            name   = CLASS_NAMES[cls_id] if cls_id < len(CLASS_NAMES) else f'class_{cls_id}'
            m      = (cv2.resize(mask, (image.shape[1], image.shape[0])) > 0.5).astype(np.uint8)
            detections.append({'name': name, 'conf': float(box.conf),
                                'mask': m, 'area_px': int(np.sum(m))})
    all_results.append({'path': img_path, 'image': image,
                         'detections': detections, 'annotated': result.plot(boxes=False)})
    det_str = [f"{d['name']} {d['conf']:.0%}" for d in detections]
    print(f'  {img_path.name}: {det_str}')

# Quick visualisation
for r in all_results:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].imshow(cv2.cvtColor(r['image'], cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Original: {r['path'].name}", fontsize=9)
    axes[0].axis('off')
    axes[1].imshow(cv2.cvtColor(r['annotated'], cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Segmentation ({len(r['detections'])} detections)")
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

## 7. Depth Estimation (MiDaS v2.1 Small)

In [ ]:
depth_device    = 'cuda' if torch.cuda.is_available() else 'cpu'
depth_model     = torch.hub.load('intel-isl/MiDaS', 'MiDaS_small', pretrained=True)
depth_model.to(depth_device).eval()
depth_transform = torch.hub.load('intel-isl/MiDaS', 'transforms').small_transform
print(f'MiDaS loaded on {depth_device}')

def get_depth_map(image_bgr):
    """Return raw MiDaS inverse-depth map (higher value = closer to camera)."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        pred = depth_model(depth_transform(rgb).to(depth_device))
        pred = torch.nn.functional.interpolate(
            pred.unsqueeze(1), size=image_bgr.shape[:2],
            mode='bicubic', align_corners=False
        ).squeeze().cpu().numpy()
    return pred   # raw inverse depth — do NOT normalise here

## 8. v3.1 Fusion Pipeline — Constants and Helper Functions

All functions updated from v2.6:
`clean_food_mask` (fallback chain), `calibrate_depth` (dual-mode),
`detect_plate`, `estimate_weight` (enriched output + per-detection mask),
`visualize_debug`.

In [ ]:
# ================================================================
# Calibration constants  (set once for your camera + plate)
# ================================================================
PLATE_DIAMETER_CM        = 25.0       # thesis FR-05
PLATE_RADIUS_CM          = PLATE_DIAMETER_CM / 2.0
PLATE_AREA_CM2           = np.pi * PLATE_RADIUS_CM ** 2   # ~490.87 cm2
EXPECTED_MAX_FOOD_HEIGHT_CM = 5.0     # assumed max food height for relative-mode scaling (not measured)

# ================================================================
# Dual-mode depth calibration (v3.1)
# ================================================================
# Set to True when running on images from the controlled Pi setup
# (fixed camera height, known plate). False for web-scraped / uncalibrated.
USE_CALIBRATED_DEPTH = False

# Populated by running calibrate_depth(). Represents how many MiDaS
# depth units correspond to 1 cm of real height at the plate surface.
DEPTH_UNITS_PER_CM = None

# When using calibrated mode, use the actual measured plate diameter.
CALIBRATED_PLATE_DIAMETER_CM = None



DEFAULT_DENSITY = 0.60   # fallback for foods not in NUTRITION_DB (approximate)


# ----------------------------------------------------------------
def clean_food_mask(raw_mask, raw_depth, plate_baseline_depth, erode_px=2,
                    min_pixels=10):
    """
    Tighten a YOLO segmentation mask. Fallback chain:
      1. eroded + depth-filtered  (best quality)
      2. raw mask + depth-filtered  (if eroded yields < min_pixels)
      3. eroded mask only  (if depth filter kills everything)
      4. raw mask  (last resort for tiny/thin foods)

    Returns (cleaned_mask: bool ndarray, cleanup_method: str).
    """
    mask_bool = (raw_mask > 0).astype(np.uint8)

    # Stage 1: morphological erosion to remove edge bleed
    eroded = mask_bool.copy()
    if erode_px > 0:
        k      = erode_px * 2 + 1
        kernel = np.ones((k, k), np.uint8)
        eroded = cv2.erode(mask_bool, kernel, iterations=1)

    # Stage 2: depth-based filtering (keep pixels above plate surface)
    tolerance   = max(plate_baseline_depth * 0.02, 1.0)
    above_plate = (raw_depth > plate_baseline_depth + tolerance).astype(np.uint8)

    # Fallback chain
    full_clean = (eroded & above_plate).astype(bool)
    if np.sum(full_clean) >= min_pixels:
        return full_clean, 'full'

    raw_depth_only = (mask_bool & above_plate).astype(bool)
    if np.sum(raw_depth_only) >= min_pixels:
        return raw_depth_only, 'uneroded_depth'

    eroded_bool = eroded.astype(bool)
    if np.sum(eroded_bool) >= min_pixels:
        return eroded_bool, 'eroded_only'

    return mask_bool.astype(bool), 'raw'


def estimate_diameter_from_mask(mask):
    """Estimate plate diameter in pixels from a binary mask."""
    contours, _ = cv2.findContours(
        mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    if not contours:
        return float(np.sqrt(max(np.sum(mask), 1) / np.pi) * 2)
    largest = max(contours, key=cv2.contourArea)
    _, radius = cv2.minEnclosingCircle(largest)
    return float(2 * radius)


# ----------------------------------------------------------------
def calibrate_depth(image, detections, raw_depth, plate_baseline,
                    object_class_name, known_object_height_cm, debug=True):
    """
    One-time calibration using a known object on the plate.

    Place an object of known height (e.g., a boiled egg on its end ~4.5 cm)
    on the plate. Run YOLO to segment it. This measures the MiDaS depth
    delta and derives DEPTH_UNITS_PER_CM.

    Returns: depth_units_per_cm (float)
    """
    obj_det = [d for d in detections if d['name'] == object_class_name]
    if not obj_det:
        raise ValueError(
            f"'{object_class_name}' not found in detections. "
            f"Available: {[d['name'] for d in detections]}"
        )

    obj_mask  = obj_det[0]['mask'] > 0
    obj_depth = raw_depth[obj_mask]
    delta_d   = obj_depth - plate_baseline
    positive  = delta_d[delta_d > 0]

    if len(positive) < 10:
        raise ValueError(
            f"Only {len(positive)} pixels above plate baseline for "
            f"'{object_class_name}'. Object may not be visible."
        )

    # Use median of top quartile as representative peak height
    top_quartile       = positive[positive >= np.percentile(positive, 75)]
    representative     = float(np.median(top_quartile))
    depth_units_per_cm = representative / known_object_height_cm

    if debug:
        print(f"  [CALIB] Object: {object_class_name}")
        print(f"  [CALIB] Pixels above plate: {len(positive)}")
        print(f"  [CALIB] Depth delta (top-Q median): {representative:.1f}")
        print(f"  [CALIB] Known height: {known_object_height_cm:.1f} cm")
        print(f"  [CALIB] >>> DEPTH_UNITS_PER_CM = {depth_units_per_cm:.2f}")
        print(f"  [CALIB] To use: set DEPTH_UNITS_PER_CM = {depth_units_per_cm:.2f}")
        print(f"  [CALIB]         set USE_CALIBRATED_DEPTH = True")

    return depth_units_per_cm


def validate_calibration(depth_units_per_cm, debug=True):
    """Sanity check on calibrated DEPTH_UNITS_PER_CM (plausible: 15–120)."""
    if depth_units_per_cm is None:
        if debug: print("  [VALID] DEPTH_UNITS_PER_CM is None — run calibrate_depth() first.")
        return False
    ok = 15 <= depth_units_per_cm <= 120
    if debug:
        status = 'PASS' if ok else 'WARNING (outside 15–120)'
        print(f"  [VALID] DEPTH_UNITS_PER_CM = {depth_units_per_cm:.2f} — {status}")
    return ok


# ----------------------------------------------------------------
def detect_plate(image, detections, raw_depth, debug=False):
    """
    3-tier plate detection: YOLO class -> Hough circles -> depth histogram.
    Returns: plate_mask, plate_baseline, px_per_cm, method.
    """
    img_h, img_w = image.shape[:2]
    min_dim = min(img_h, img_w)

    # Tier 1: YOLO plate class
    if PLATE_CLASS is not None:
        for det in detections:
            if det['name'] == PLATE_CLASS:
                plate_mask = det['mask'].astype(bool)
                if np.sum(plate_mask) > 100:
                    plate_baseline = float(np.median(raw_depth[plate_mask]))
                    diam_px   = estimate_diameter_from_mask(det['mask'])
                    px_per_cm = diam_px / PLATE_DIAMETER_CM
                    if debug:
                        print(f'  [PLATE] YOLO: px/cm={px_per_cm:.2f}')
                    return plate_mask, plate_baseline, px_per_cm, 'yolo'

    # Tier 2: Hough circles
    gray    = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (9, 9), 2)
    circles = cv2.HoughCircles(
        blurred, cv2.HOUGH_GRADIENT, dp=1.2,
        minDist=min_dim // 2, param1=50, param2=30,
        minRadius=min_dim // 6, maxRadius=min_dim // 2,
    )
    if circles is not None:
        circles = np.round(circles[0]).astype(int)
        cx, cy, r = max(circles, key=lambda c: c[2])
        px_per_cm = (2 * r) / PLATE_DIAMETER_CM
        if debug:
            print(f'  [PLATE] Hough r={r}px -> px/cm={px_per_cm:.2f}')
        plate_mask_u8 = np.zeros((img_h, img_w), dtype=np.uint8)
        cv2.circle(plate_mask_u8, (int(cx), int(cy)), int(r), 1, -1)
        plate_mask     = plate_mask_u8.astype(bool)
        plate_baseline = float(np.median(raw_depth[plate_mask]))
        return plate_mask, plate_baseline, px_per_cm, 'hough'
    elif debug:
        print('  [PLATE] Hough found no circles, using depth_fallback')

    # Tier 3: depth histogram mode
    hist, bin_edges = np.histogram(raw_depth.flatten(), bins=256)
    mode_idx   = int(np.argmax(hist))
    mode_depth = float((bin_edges[mode_idx] + bin_edges[mode_idx + 1]) / 2)
    if mode_depth < 1e-6:
        mode_depth = float(np.median(raw_depth))
    plate_mask     = np.abs(raw_depth - mode_depth) < (mode_depth * 0.05)
    plate_baseline = mode_depth
    radius_px  = float(np.sqrt(max(np.sum(plate_mask), 1) / np.pi))
    px_per_cm  = max((2 * radius_px) / PLATE_DIAMETER_CM, 1.0)
    if debug:
        print(f'  [PLATE] depth_fallback: mode={mode_depth:.1f}, px/cm={px_per_cm:.2f}')
    return plate_mask, plate_baseline, px_per_cm, 'depth_fallback'


# ----------------------------------------------------------------
def estimate_weight(image, detections, debug=False):
    """
    Fuse segmentation masks with MiDaS depth to estimate food weight.

    Returns: foods (list of enriched dicts), info (dict with intermediates).

    Each food dict contains:
        class_name, conf, mask, area_cm2, height_cm, volume_cm3, weight_g,
        gi_value, gi_class, carbs_g, cleanup_method.
    """
    raw_depth = get_depth_map(image)
    plate_mask, plate_baseline, px_per_cm, plate_method = detect_plate(
        image, detections, raw_depth, debug=debug
    )
    if USE_CALIBRATED_DEPTH and CALIBRATED_PLATE_DIAMETER_CM is not None:
        diam_px   = px_per_cm * PLATE_DIAMETER_CM
        px_per_cm = diam_px / CALIBRATED_PLATE_DIAMETER_CM
    pixel_area_cm2 = (1.0 / px_per_cm) ** 2

    # Pass 1: clean masks, collect depth deltas for normalisation
    cleaned_detections  = []
    all_positive_deltas = []
    for d in detections:
        if (PLATE_CLASS is not None and d['name'] == PLATE_CLASS) or \
           d['name'].lower() == 'plate':
            continue
        if int(np.sum(d['mask'] > 0)) == 0:
            continue

        cleaned, cleanup_method = clean_food_mask(
            d['mask'], raw_depth, plate_baseline, erode_px=2
        )
        n_pixels = int(np.sum(cleaned))

        if n_pixels < 1:
            if debug:
                print(f'  [SKIP] {d["name"]}: 0 px after all fallbacks')
            continue

        cleaned_detections.append((d, cleaned, cleanup_method))
        pos = (raw_depth[cleaned] - plate_baseline)
        pos = pos[pos > 0]
        if len(pos) > 0:
            all_positive_deltas.append(pos)

    delta_95 = (
        float(np.percentile(np.concatenate(all_positive_deltas), 95))
        if all_positive_deltas else 1.0
    )

    if debug:
        print(f'  [PLATE] method={plate_method} | px/cm={px_per_cm:.2f} | '
              f'baseline={plate_baseline:.1f} | pixel_area={pixel_area_cm2:.5f} cm2')
        if USE_CALIBRATED_DEPTH and DEPTH_UNITS_PER_CM is not None:
            print(f'  [DEPTH] CALIBRATED mode | units/cm={DEPTH_UNITS_PER_CM:.2f}')
        else:
            print(f'  [DEPTH] RELATIVE mode | delta_95={delta_95:.1f} | '
                  f'max_h={EXPECTED_MAX_FOOD_HEIGHT_CM} cm')

    height_map = np.zeros(image.shape[:2], dtype=np.float32)
    foods      = []

    # Pass 2: compute heights and volumes
    for d, cleaned, cleanup_method in cleaned_detections:
        delta_d = raw_depth[cleaned] - plate_baseline

        # Dual-mode height computation (v3.1)
        if USE_CALIBRATED_DEPTH and DEPTH_UNITS_PER_CM is not None:
            # CALIBRATED MODE: direct physical conversion
            height_cm = np.clip(delta_d / DEPTH_UNITS_PER_CM, 0.0, None)
            norm_mean = float(np.mean(height_cm)) / EXPECTED_MAX_FOOD_HEIGHT_CM
        else:
            # RELATIVE MODE: percentile-based normalisation
            normalized_h = np.clip(delta_d / max(delta_95, 1e-6), 0.0, 1.5)
            height_cm    = np.clip(normalized_h * EXPECTED_MAX_FOOD_HEIGHT_CM, 0.0, None)
            norm_mean = float(np.mean(normalized_h))

        volume_cm3  = float(np.sum(height_cm) * pixel_area_cm2)
        area_cm2    = float(np.sum(cleaned) * pixel_area_cm2)
        mean_height = float(np.mean(height_cm))

        # Density from unified NUTRITION_DB (v3.1 — no more FOOD_DENSITIES)
        _db_entry = NUTRITION_DB.get(db_key(d['name']), {})
        density   = _db_entry.get('density', DEFAULT_DENSITY)
        weight_g  = volume_cm3 * density
        height_map[cleaned] = height_cm

        # Nutritional enrichment for recommendation engine (v3.1)
        _per100  = _db_entry.get('per_100g', {})
        gi_value = _db_entry.get('glycemic_index')
        gi_class = _db_entry.get('gi_classification')
        carbs_g  = (weight_g / 100.0) * _per100.get('carbs', 0)

        flag      = ''
        if debug:
            print(f'  [{d["name"]:<22}] ({cleanup_method:<14}) '
                  f'norm_h={norm_mean:.2f} | '
                  f'mean_h={mean_height:.2f} cm{flag} | '
                  f'vol={volume_cm3:.1f} cm3 | wt={weight_g:.1f}g')

        foods.append({
            'class_name':     d['name'],
            'conf':           d['conf'],
            'mask':           cleaned,
            'cleanup_method': cleanup_method,
            'area_cm2':       round(area_cm2, 1),
            'height_cm':      round(mean_height, 2),
            'volume_cm3':     round(volume_cm3, 1),
            'weight_g':       round(weight_g, 1),
            'gi_value':       gi_value,
            'gi_class':       gi_class,
            'carbs_g':        round(carbs_g, 1),
        })

    vis_depth = (raw_depth - raw_depth.min()) / (raw_depth.max() - raw_depth.min() + 1e-8)
    info = {
        'vis_depth': vis_depth, 'raw_depth': raw_depth,
        'px_per_cm': px_per_cm, 'plate_method': plate_method,
        'plate_mask': plate_mask, 'plate_baseline': plate_baseline,
        'height_map': height_map, 'pixel_area_cm2': pixel_area_cm2,
        'delta_95': delta_95,
    }
    return foods, info


# ----------------------------------------------------------------
def visualize_debug(image, info):
    """4-panel diagnostic: Input | Raw Depth | Plate Mask | Height Map."""
    raw_depth      = info['raw_depth']
    plate_mask     = info['plate_mask']
    plate_baseline = info['plate_baseline']
    height_map     = info['height_map']
    plate_method   = info['plate_method']
    px_per_cm      = info['px_per_cm']
    delta_95       = info['delta_95']

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes[0, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0, 0].set_title('Input Image'); axes[0, 0].axis('off')
    im0 = axes[0, 1].imshow(raw_depth, cmap='plasma')
    axes[0, 1].set_title(
        f'MiDaS Raw Depth  (baseline={plate_baseline:.1f}, delta_95={delta_95:.1f})'
    ); axes[0, 1].axis('off')
    plt.colorbar(im0, ax=axes[0, 1], fraction=0.046, pad=0.04)
    axes[1, 0].imshow(plate_mask, cmap='gray')
    axes[1, 0].set_title(f'Plate Mask  [{plate_method}  {px_per_cm:.2f} px/cm]')
    axes[1, 0].axis('off')
    im1 = axes[1, 1].imshow(height_map, cmap='hot', vmin=0,
                             vmax=EXPECTED_MAX_FOOD_HEIGHT_CM)
    axes[1, 1].set_title(f'Food Height Map (cm, max={EXPECTED_MAX_FOOD_HEIGHT_CM} cm)')
    axes[1, 1].axis('off')
    plt.colorbar(im1, ax=axes[1, 1], fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()


print('v3.2 pipeline functions loaded: clean_food_mask, calibrate_depth, '
      'detect_plate, estimate_weight, visualize_debug')

## 9. Nutrition Database (unified density source)

Expanded to cover all 16 YOLO classes. Each entry has:
`glycemic_index`, `gi_classification` (Low / Medium / High),
`per_100g` macros, and `density` (g/cm³) for volume-to-weight conversion.

`FOOD_DENSITIES` has been removed; density is read directly from this DB.

In [ ]:
# Keys are lowercase YOLO class names (db_key() normalises Title_Case -> lower)
# GDA serving weights (gda_serving_g): grams providing 20g carbs.
# Source: Ghana Dietetic Association Standard Serving Sizes (Lartey et al. 1999).
NUTRITION_DB = {
    'jollof_rice':     {'name': 'Jollof Rice',     'per_100g': {'calories': 175, 'carbs': 32,  'protein': 4.5, 'fat': 3.2,  'fiber': 1.2}, 'glycemic_index': 70, 'gi_classification': 'High',   'density': 0.55}, 'gda_serving_g': 73,
    'waakye':          {'name': 'Waakye',           'per_100g': {'calories': 165, 'carbs': 28,  'protein': 6.5, 'fat': 2.0,  'fiber': 4.5}, 'glycemic_index': 55, 'gi_classification': 'Medium', 'density': 0.60}, 'gda_serving_g': 77,
    'banku':           {'name': 'Banku',            'per_100g': {'calories': 165, 'carbs': 28,  'protein': 2.1, 'fat': 0.5,  'fiber': 0.8}, 'glycemic_index': 73, 'gi_classification': 'High',   'density': 0.50}, 'gda_serving_g': 84,  # GI: Eli-Cophie et al. 2017
    'fufu':            {'name': 'Fufu',             'per_100g': {'calories': 170, 'carbs': 40,  'protein': 1.5, 'fat': 0.3,  'fiber': 1.0}, 'glycemic_index': 55, 'gi_classification': 'Low',    'density': 0.55}, 'gda_serving_g': 62,  # GI: Eli-Cophie et al. 2017
    'plain_rice':      {'name': 'Plain Rice',       'per_100g': {'calories': 130, 'carbs': 28,  'protein': 2.7, 'fat': 0.3,  'fiber': 0.4}, 'glycemic_index': 73, 'gi_classification': 'High',   'density': 0.55}, 'gda_serving_g': 72,
    'fried_plantain':  {'name': 'Fried Plantain',   'per_100g': {'calories': 250, 'carbs': 35,  'protein': 1.2, 'fat': 12.0, 'fiber': 2.3}, 'glycemic_index': 70, 'gi_classification': 'High',   'density': 0.45}, 'gda_serving_g': 39,
    'grilled_chicken': {'name': 'Grilled Chicken',  'per_100g': {'calories': 165, 'carbs': 0,   'protein': 31,  'fat': 3.6,  'fiber': 0},   'glycemic_index': 0,  'gi_classification': 'Low',    'density': 0.85}, 'gda_serving_g': None,
    'tilapia':         {'name': 'Tilapia',          'per_100g': {'calories': 130, 'carbs': 0,   'protein': 26,  'fat': 2.7,  'fiber': 0},   'glycemic_index': 0,  'gi_classification': 'Low',    'density': 0.80}, 'gda_serving_g': None,
    'fried_fish':      {'name': 'Fried Fish',       'per_100g': {'calories': 190, 'carbs': 5,   'protein': 20,  'fat': 10.0, 'fiber': 0},   'glycemic_index': 0,  'gi_classification': 'Low',    'density': 0.75}, 'gda_serving_g': None,
    'beans':           {'name': 'Beans',            'per_100g': {'calories': 132, 'carbs': 22,  'protein': 8.7, 'fat': 0.5,  'fiber': 6.4}, 'glycemic_index': 28, 'gi_classification': 'Low',    'density': 0.65}, 'gda_serving_g': 115,
    'boiled_egg':      {'name': 'Boiled Egg',       'per_100g': {'calories': 155, 'carbs': 1.1, 'protein': 13,  'fat': 11,   'fiber': 0},   'glycemic_index': 0,  'gi_classification': 'Low',    'density': 1.03}, 'gda_serving_g': None,
    'okro_soup':       {'name': 'Okro Soup',        'per_100g': {'calories': 60,  'carbs': 7,   'protein': 3.0, 'fat': 2.5,  'fiber': 3.0}, 'glycemic_index': 20, 'gi_classification': 'Low',    'density': 0.90}, 'gda_serving_g': None,
    'light_soup':      {'name': 'Light Soup',       'per_100g': {'calories': 45,  'carbs': 5,   'protein': 3.0, 'fat': 2.0,  'fiber': 1.0}, 'glycemic_index': 20, 'gi_classification': 'Low',    'density': 0.95}, 'gda_serving_g': None,
    'salad':           {'name': 'Salad',            'per_100g': {'calories': 20,  'carbs': 3,   'protein': 1.5, 'fat': 0.3,  'fiber': 2.0}, 'glycemic_index': 15, 'gi_classification': 'Low',    'density': 0.30}, 'gda_serving_g': None,
    'shito':           {'name': 'Shito',            'per_100g': {'calories': 180, 'carbs': 8,   'protein': 5.0, 'fat': 15,   'fiber': 2.0}, 'glycemic_index': 10, 'gi_classification': 'Low',    'density': 0.90}, 'gda_serving_g': None,
}

def db_key(yolo_class_name: str) -> str:
    """Normalise a YOLO class name to its nutrition-DB key.
    e.g. 'Jollof_Rice' -> 'jollof_rice'
    """
    return yolo_class_name.lower()

print(f'Nutrition DB loaded: {len(NUTRITION_DB)} foods')

## 10. Diabetic Plate Model v3.1 — Constants

Clinical framework from dietician consultation:
- **50 %** non-starchy vegetables / fibre
- **25 %** protein
- **25 %** starchy carbohydrates

**Categories:**
- `starch_moulded` — fufu, banku. Orange volume reference applies.
- `starch_spread` — jollof rice, waakye, fried plantain. Plate ratio only.
- `protein` — chicken, tilapia, fish, egg, beans.
- `vegetable` — salad.
- `soup_sauce` — okro soup, light soup, shito. Excluded from plate ratio.

Starch portion benchmark: a **medium orange (~220 cm³)**.

In [ ]:
# Maps YOLO class names to plate categories.
# Categories:
#   starch_moulded  — fufu, banku. Orange volume reference applies.
#   starch_spread   — jollof rice, waakye, fried plantain, etc. Plate ratio only.
#   protein         — chicken, tilapia, fish, egg, beans.
#   vegetable       — salad.
#   soup_sauce      — okro soup, light soup, shito. Excluded from plate ratio.
PLATE_CATEGORY = {
    'Jollof_Rice':     'starch_spread',
    'Waakye':          'starch_spread',    # rice-and-beans staple
    'Plain_Rice':      'starch_spread',
    'Fried_Plantain':  'starch_spread',
    'Banku':           'starch_moulded',
    'Fufu':            'starch_moulded',
    'Beans':           'protein',          # legume — protein source
    'Grilled_Chicken': 'protein',
    'Tilapia':         'protein',
    'Fried_Fish':      'protein',
    'Boiled_Egg':      'protein',
    'Okro_Soup':       'soup_sauce',       # accompaniment, not standalone veg
    'Light_Soup':      'soup_sauce',       # broth — meat/fish detected separately
    'Shito':           'soup_sauce',       # pepper sauce with fish/shrimp
    'Salad':           'vegetable',
    # 'Plate' is excluded — it is the reference surface, not food
}

# Dietician benchmark: medium orange (~7-8 cm diameter) ~ 220 cm3.
# Used ONLY for starch_moulded staples (fufu, banku).
STARCH_REFERENCE_VOLUME_CM3 = 220.0

PORTION_THRESHOLDS = {
    'small':       0.50,   # below 50 % of reference
    'appropriate': 1.00,   # up to 100 % (inclusive)
    'reduce':      1.50,   # up to 150 %
    # above 150 % -> 'excessive'
}

# LCD 16-char x 2-line display strings
# GDA serving thresholds (number of servings, where 1 serving = 20g carbs).
# Source: Ghana Dietetic Association Standard Serving Sizes (Lartey et al. 1999).
GDA_SPREAD_THRESHOLDS = {
    'small':       0.75,
    'appropriate': 2.00,   # up to 2 servings = ~40g carbs from starch
    'reduce':      3.00,
}

LCD_MESSAGES = {
    'good':    ('Plate Balanced',  'Good portion!'),
    'caution': ('Add Vegetables',  'or Reduce starch'),
    'warning': ('Too Much Starch', 'Reduce portion!'),
}

print('v3.3 Diabetic Plate Model constants loaded.')
print(f'  PLATE_CATEGORY covers {len(PLATE_CATEGORY)} food classes')
print(f'  Starch reference: {STARCH_REFERENCE_VOLUME_CM3:.0f} cm3 (medium orange)')

## 11. v3.1 Recommendation Engine

Three functions, two levels:

| Function | Input | Logic |
|---|---|---|
| `classify_plate_composition` | mask pixel areas | 50/25/25 ratio check (starch = moulded + spread) |
| `classify_starch_portion` | volume_cm3 | compare moulded starch vs 220 cm³ orange; spread assessed by ratio only |
| `generate_recommendation` | both above | two-level alert decision |

`soup_sauce` items are excluded from plate ratio calculation.
Spread starches are assessed by plate ratio only (not volume-compared to orange).

In [ ]:
def classify_plate_composition(detections, plate_area_px):
    """
    Assess plate composition against the 50/25/25 Diabetic Plate Model.

    Uses mask pixel AREAS (not volume) for ratio computation.
    Both starch subtypes (moulded + spread) count as 'starch'.
    soup_sauce items are excluded from the ratio (accompaniments).
    """
    category_pixels = {'starch': 0, 'protein': 0, 'vegetable': 0}

    for det in detections:
        cat       = PLATE_CATEGORY.get(det['class_name'], 'unknown')
        mask_area = int(det['mask'].sum())
        if cat in ('starch_moulded', 'starch_spread'):
            category_pixels['starch'] += mask_area
        elif cat == 'soup_sauce':
            pass  # excluded from plate ratio — soups/sauces are accompaniments
        elif cat == 'mixed':
            category_pixels['starch']  += mask_area // 2
            category_pixels['protein'] += mask_area // 2
        elif cat in category_pixels:
            category_pixels[cat] += mask_area

    total_food_px = sum(category_pixels.values())
    if total_food_px == 0:
        return {
            'ratios':         {'starch': 0.0, 'protein': 0.0, 'vegetable': 0.0},
            'plate_balanced': False,
            'vegetables_low': True,
            'messages':       ['No food detected on plate.'],
        }

    ratios = {k: v / total_food_px for k, v in category_pixels.items()}

    vegetables_low = ratios['vegetable'] < 0.30
    plate_balanced = ratios['vegetable'] >= 0.40 and ratios['starch'] <= 0.35

    messages = []
    if vegetables_low:
        messages.append(
            'Add more vegetables to your plate for better blood sugar control.'
        )
    if ratios['starch'] > 0.40:
        messages.append(
            'Your plate has too much starchy food. '
            'Try to keep starches to about a quarter of your plate.'
        )
    return {
        'ratios': ratios, 'plate_balanced': plate_balanced,
        'vegetables_low': vegetables_low, 'messages': messages,
    }


def classify_starch_portion(detections):
    """
    Classify starch portions using two reference systems (v3.3).

    - starch_moulded (banku, fufu): volume vs 220 cm³ medium orange.
    - starch_spread (jollof, rice, plantain): weight vs GDA serving sizes.
    """
    moulded_volume        = 0.0
    moulded_foods         = []
    spread_weight         = 0.0
    spread_foods          = []
    spread_servings_total = 0.0

    for det in detections:
        cat = PLATE_CATEGORY.get(det['class_name'], 'unknown')
        if cat == 'starch_moulded':
            moulded_volume += det['volume_cm3']
            moulded_foods.append(det['class_name'])
        elif cat == 'starch_spread':
            w = det.get('weight_g', 0.0)
            spread_weight += w
            spread_foods.append(det['class_name'])
            db_entry = NUTRITION_DB.get(db_key(det['class_name']), {})
            gda_g = db_entry.get('gda_serving_g')
            if gda_g and gda_g > 0:
                spread_servings_total += w / gda_g
        elif cat == 'mixed':
            moulded_volume += det['volume_cm3'] * 0.5

    # ── Path 1: Moulded starches (volume-based, orange reference) ───────────
    if moulded_volume > 0:
        ratio = moulded_volume / STARCH_REFERENCE_VOLUME_CM3
        if   ratio <= PORTION_THRESHOLDS['small']:       category = 'small'
        elif ratio <= PORTION_THRESHOLDS['appropriate']:  category = 'appropriate'
        elif ratio <= PORTION_THRESHOLDS['reduce']:       category = 'reduce'
        else:                                             category = 'excessive'

        food_label = ', '.join(f.replace('_', ' ').title() for f in moulded_foods)
        _MOULDED_MESSAGES = {
            'small':       f'Your {food_label} portion is small.',
            'appropriate': f'Your {food_label} portion looks good.',
            'reduce':      (f'Consider reducing your {food_label}. '
                            'Aim for about the size of a medium orange.'),
            'excessive':   (f'Your {food_label} portion is too large. '
                            'It should be about the size of a medium orange.'),
        }
        return {
            'total_starch_volume_cm3': moulded_volume,
            'total_starch_weight_g':   0.0,
            'portion_category':        category,
            'ratio_to_reference':      round(ratio, 2),
            'reference_type':          'volume',
            'message':                 _MOULDED_MESSAGES[category],
        }

    # ── Path 2: Spread starches (weight-based, GDA serving reference) ─────
    if spread_weight > 0:
        if   spread_servings_total <= GDA_SPREAD_THRESHOLDS['small']:       category = 'small'
        elif spread_servings_total <= GDA_SPREAD_THRESHOLDS['appropriate']:  category = 'appropriate'
        elif spread_servings_total <= GDA_SPREAD_THRESHOLDS['reduce']:       category = 'reduce'
        else:                                                                category = 'excessive'

        food_label   = ', '.join(f.replace('_', ' ').title() for f in spread_foods)
        servings_str = f'{spread_servings_total:.1f}'
        _SPREAD_MESSAGES = {
            'small':       f'Your {food_label} portion is small ({servings_str} servings).',
            'appropriate': f'Your {food_label} portion is appropriate ({servings_str} servings).',
            'reduce':      (f'Consider reducing your {food_label} ({servings_str} servings). '
                            'Aim for about 2 servings per meal.'),
            'excessive':   (f'Your {food_label} portion is too large ({servings_str} servings). '
                            'A single meal should have about 2 servings.'),
        }
        return {
            'total_starch_volume_cm3': 0.0,
            'total_starch_weight_g':   round(spread_weight, 1),
            'portion_category':        category,
            'ratio_to_reference':      round(spread_servings_total, 2),
            'reference_type':          'weight',
            'message':                 _SPREAD_MESSAGES[category],
        }

    # ── Path 3: No starch detected ─────────────────────────────
    return {
        'total_starch_volume_cm3': 0.0,
        'total_starch_weight_g':   0.0,
        'portion_category':        'none',
        'ratio_to_reference':      0.0,
        'reference_type':          'none',
        'message':                 'No starchy food detected.',
    }


def generate_recommendation(detections, plate_area_px):
    """
    Top-level v3.1 recommendation engine.

    Combines plate-composition check (area-based) with starch-portion
    check (volume-based) into a single alert + message set.

    detections   : list of dicts with class_name, mask, volume_cm3,
                   weight_g, gi_value, gi_class, carbs_g.
    plate_area_px: total plate pixel area from info['plate_mask'].

    Returns dict:
        plate_assessment, starch_assessment, gi_info,
        overall_message, detail_messages, alert_level.
    """
    plate  = classify_plate_composition(detections, plate_area_px)
    starch = classify_starch_portion(detections)

    gi_info = [
        {'food': d['class_name'], 'gi': d.get('gi_value'), 'gi_class': d.get('gi_class')}
        for d in detections if d.get('gi_value') is not None
    ]

    detail_messages = []

    # ── Two-level decision ────────────────────────────────────────────
    if starch['portion_category'] == 'excessive':
        alert_level     = 'warning'
        overall_message = starch['message']
        detail_messages.extend(plate['messages'])

    elif starch['portion_category'] == 'reduce':
        alert_level     = 'caution'
        overall_message = starch['message']
        detail_messages.extend(plate['messages'])

    else:
        # Starch is fine (or none); check plate composition
        if plate['vegetables_low']:
            alert_level     = 'caution'
            overall_message = ('Add more vegetables to your plate '
                               'for better blood sugar control.')
            if starch['portion_category'] == 'small':
                detail_messages.append(
                    "Your starch portion is small — that's fine, "
                    'just fill the gap with vegetables.'
                )
            else:
                detail_messages.append(starch['message'])
        else:
            alert_level     = 'good'
            overall_message = 'Your plate looks well balanced.'
            detail_messages.append(starch['message'])

    # ── High-GI context ───────────────────────────────────────────────
    high_gi = [g for g in gi_info if g['gi_class'] == 'High']
    if high_gi:
        names = ', '.join(g['food'].replace('_', ' ').title() for g in high_gi)
        detail_messages.append(
            f'{names} has a high glycemic index. '
            'Pair with fibre or vegetables to slow glucose absorption.'
        )

    return {
        'plate_assessment':  plate,
        'starch_assessment': starch,
        'gi_info':           gi_info,
        'overall_message':   overall_message,
        'detail_messages':   detail_messages,
        'alert_level':       alert_level,
    }


print('v3.3 functions loaded: classify_plate_composition, '
      'classify_starch_portion, generate_recommendation')

## 12. Full Inference + Dietician Recommendations (v3.1)

Runs the complete pipeline on each validation image:
segmentation → depth → volume/weight → plate assessment → recommendation.

`estimate_weight` now returns enriched dicts with mask, GI, and carbs —
no bridging `name_to_mask` dict needed.

In [ ]:
ALERT_LABEL  = {'good': '[GOOD]', 'caution': '[CAUTION]', 'warning': '[WARNING]'}
ALERT_COLOR  = {'good': 'lime',   'caution': 'yellow',    'warning': 'red'}

for r in all_results:
    image      = r['image']
    detections = r['detections']

    print(f"\n{'='*72}")
    print(f"Image: {r['path'].name}")

    # ── Step 1: volume + weight (v3.1 pipeline — enriched output) ─────
    foods, info = estimate_weight(image, detections, debug=True)

    if not foods:
        print('  No food detected after mask cleaning — skipping.')
        continue

    # ── Step 2: pass enriched foods directly to recommendation engine ─
    # No name_to_mask bridge needed — each food carries its own mask,
    # class_name, gi_value, gi_class, and carbs_g (v3.1).
    plate_area_px = int(np.sum(info['plate_mask'])) if info['plate_mask'] is not None else 0
    rec = generate_recommendation(foods, plate_area_px)

    # ── Step 3: display ───────────────────────────────────────────────
    # Weight + GI table
    print(f"\n  {'Food':<24} {'Wt g':>6} {'Vol cm3':>8} {'Carbs g':>8} "
          f"{'GI':>5} {'GI Class':<8}")
    print('  ' + '-' * 62)
    for food in foods:
        gi  = food.get('gi_value', '--')
        gic = food.get('gi_class', '--')
        cat = PLATE_CATEGORY.get(food['class_name'], '?')
        print(f"  {food['class_name']:<24} {food['weight_g']:>6.0f} "
              f"{food['volume_cm3']:>8.1f} {food['carbs_g']:>8.1f} "
              f"{str(gi or '--'):>5} {gic or '--':<8}  [{cat}]")

    # Recommendation panel
    ra  = rec['plate_assessment']['ratios']
    sa  = rec['starch_assessment']
    lcd = LCD_MESSAGES[rec['alert_level']]

    print(f"\n  {'─'*55}")
    print(f"  DIABETIC PLATE ASSESSMENT (v3.3)")
    print(f"  {'─'*55}")
    print(f"  {ALERT_LABEL[rec['alert_level']]}  {rec['overall_message']}")
    for msg in rec['detail_messages']:
        print(f'    -> {msg}')
    print(f"\n  Plate composition (by mask area):")
    print(f"    Starch    {ra['starch']:>5.0%}   target ~25%")
    print(f"    Protein   {ra['protein']:>5.0%}   target ~25%")
    print(f"    Vegetable {ra['vegetable']:>5.0%}   target ~50%")
    if sa['portion_category'] != 'none':
        if sa.get('reference_type') == 'volume':
            print(f"\n  Starch volume : {sa['total_starch_volume_cm3']:.0f} cm3  "
                  f"({sa['ratio_to_reference']:.1f}x orange reference "
                  f"of {STARCH_REFERENCE_VOLUME_CM3:.0f} cm3)")
        elif sa.get('reference_type') == 'weight':
            print(f"\n  Starch weight : {sa['total_starch_weight_g']:.0f} g  "
                  f"({sa['ratio_to_reference']:.1f} GDA servings)")
        print(f"  Portion size  : {sa['portion_category'].upper()}")
    print(f"\n  LCD: [{lcd[0]:<16}] / [{lcd[1]:<16}]")

    # ── Visualisation ─────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original'); axes[0].axis('off')

    axes[1].imshow(cv2.cvtColor(r['annotated'], cv2.COLOR_BGR2RGB))
    axes[1].set_title('Segmentation'); axes[1].axis('off')
    axes[1].text(4, 18, ALERT_LABEL[rec['alert_level']],
                 color=ALERT_COLOR[rec['alert_level']], fontsize=10,
                 fontweight='bold',
                 bbox=dict(facecolor='black', alpha=0.65, pad=3))
    short_msg = rec['overall_message'][:45]
    axes[1].text(4, 38, short_msg, color='white', fontsize=7,
                 bbox=dict(facecolor='black', alpha=0.50, pad=2))

    axes[2].imshow(1 - info['vis_depth'], cmap='plasma')
    axes[2].set_title('Depth Map (dark=close)'); axes[2].axis('off')

    plt.suptitle(
        f"{r['path'].name}  |  {ALERT_LABEL[rec['alert_level']]}  "
        f"Starch {ra['starch']:.0%}  Veg {ra['vegetable']:.0%}",
        fontsize=9
    )
    plt.tight_layout(); plt.show()

    # 4-panel debug view
    visualize_debug(image, info)

## 13. Export Model (NCNN for Raspberry Pi)

In [ ]:
export_model = YOLO(MODEL_PATH)
export_path  = export_model.export(format='ncnn')
print(f'Exported to: {export_path}')

## 14. Export Pi Deployment Package to Google Drive

Mounts Google Drive and copies everything the Raspberry Pi 5 needs:
- `best.pt` — PyTorch weights (fallback / further fine-tuning)
- `best_ncnn_model/` — NCNN `.param` + `.bin` for on-device inference
- `nutrition_db.json` — nutrition lookup table (GI, density, carbs)
- `class_names.json` — YOLO class index to name mapping
- `plate_category.json` — class to plate category mapping
- `pipeline_constants.json` — calibration constants snapshot


In [ ]:
from google.colab import drive
import shutil, json, os
from pathlib import Path

# -- 1. Mount Drive -------------------------------------------------------
drive.mount('/content/drive', force_remount=False)

# -- 2. Destination folder ------------------------------------------------
DRIVE_EXPORT_DIR = Path('/content/drive/MyDrive/capstone_pi_deploy')
DRIVE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Export destination: {DRIVE_EXPORT_DIR}')

# -- 3. YOLO .pt weights --------------------------------------------------
pt_src = Path(MODEL_PATH)
if pt_src.exists():
    shutil.copy2(pt_src, DRIVE_EXPORT_DIR / 'best.pt')
    print(f'  [OK]   best.pt  ({pt_src.stat().st_size / 1e6:.1f} MB)')
else:
    print(f'  [SKIP] best.pt not found at {pt_src} -- run training first')

# -- 4. NCNN model folder (.param + .bin) ---------------------------------
ncnn_src = pt_src.parent / 'best_ncnn_model'
ncnn_dst  = DRIVE_EXPORT_DIR / 'best_ncnn_model'
if ncnn_src.exists():
    if ncnn_dst.exists():
        shutil.rmtree(ncnn_dst)
    shutil.copytree(ncnn_src, ncnn_dst)
    sizes = ', '.join(
        f'{f.name} ({f.stat().st_size/1e6:.1f} MB)'
        for f in sorted(ncnn_dst.iterdir())
    )
    print(f'  [OK]   best_ncnn_model/  {sizes}')
else:
    print(f'  [SKIP] NCNN folder not found at {ncnn_src} -- run cell 26 first')

# -- 5. Nutrition DB as JSON ----------------------------------------------
with open(DRIVE_EXPORT_DIR / 'nutrition_db.json', 'w') as fp:
    json.dump(NUTRITION_DB, fp, indent=2)
print(f'  [OK]   nutrition_db.json  ({len(NUTRITION_DB)} foods)')

# -- 6. Class names (index -> name) ---------------------------------------
with open(DRIVE_EXPORT_DIR / 'class_names.json', 'w') as fp:
    json.dump(seg_model.names, fp, indent=2)
print(f'  [OK]   class_names.json  ({len(seg_model.names)} classes)')

# -- 7. Plate category mapping --------------------------------------------
with open(DRIVE_EXPORT_DIR / 'plate_category.json', 'w') as fp:
    json.dump(PLATE_CATEGORY, fp, indent=2)
print(f'  [OK]   plate_category.json  ({len(PLATE_CATEGORY)} entries)')

# -- 8. Pipeline constants snapshot ---------------------------------------
constants = {
    'notebook_version':             'v3.3',
    'PLATE_DIAMETER_CM':            PLATE_DIAMETER_CM,
    'PLATE_AREA_CM2':               round(PLATE_AREA_CM2, 4),
    'EXPECTED_MAX_FOOD_HEIGHT_CM':  EXPECTED_MAX_FOOD_HEIGHT_CM,
    'USE_CALIBRATED_DEPTH':         USE_CALIBRATED_DEPTH,
    'DEPTH_UNITS_PER_CM':           DEPTH_UNITS_PER_CM,
    'CALIBRATED_PLATE_DIAMETER_CM': CALIBRATED_PLATE_DIAMETER_CM,
    'STARCH_REFERENCE_VOLUME_CM3':  STARCH_REFERENCE_VOLUME_CM3,
    'DEFAULT_DENSITY':              DEFAULT_DENSITY,
    'PORTION_THRESHOLDS':           PORTION_THRESHOLDS,
    'GDA_SPREAD_THRESHOLDS':        GDA_SPREAD_THRESHOLDS,
}
with open(DRIVE_EXPORT_DIR / 'pipeline_constants.json', 'w') as fp:
    json.dump(constants, fp, indent=2)
print('  [OK]   pipeline_constants.json')

# -- 9. Summary -----------------------------------------------------------
print()
print('=== Pi deployment package ===')
for item in sorted(DRIVE_EXPORT_DIR.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / 1e6
    else:
        size_mb = sum(x.stat().st_size for x in item.rglob('*') if x.is_file()) / 1e6
    print(f'  {item.name:<35s}  {size_mb:7.2f} MB')
print()
print(f'Drive path: {DRIVE_EXPORT_DIR}')
